# UTD xBD Damage From Scratch
Randomly initialized EfficientNet-B0, paired RGB crops, source-tile splits. Noncommercial research only.
Data: https://www.kaggle.com/datasets/tunguz/xview2-challenge-dataset-train-and-test (v3), CC BY-NC-SA 4.0.
Classification uses known building footprints; not a building detector, safety decision or validated emergency tool.


In [ ]:
from pathlib import Path
import sys, json, torch, torchvision, shapely
DATA_ROOT = "/kaggle/input/datasets/tunguz/xview2-challenge-dataset-train-and-test"
assert torch.cuda.is_available(), "Free GPU unavailable"
print("GPU:", torch.cuda.get_device_name(0))
print("Versions:", torch.__version__, torchvision.__version__, shapely.__version__)
Path("/kaggle/working/xbd_pipeline.py").write_text('"""Small, reproducible xBD building-crop pipeline. No network or credentials."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport random\nfrom collections import Counter\nfrom pathlib import Path\n\nCLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]\nIMAGE_SIZE = 224\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef crop_bounds(bounds, width, height, context=0.2):\n    """Use the same square box for both registered images, clipped at edges."""\n    x0, y0, x1, y1 = map(float, bounds)\n    if not all(math.isfinite(x) for x in (x0, y0, x1, y1)):\n        raise ValueError("Non-finite footprint bounds")\n    if x1 <= x0 or y1 <= y0:\n        raise ValueError("Empty footprint")\n    side = max(x1 - x0, y1 - y0, 8) * (1 + 2 * context)\n    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2\n    result = [max(0, math.floor(cx-side/2)), max(0, math.floor(cy-side/2)),\n              min(width, math.ceil(cx+side/2)), min(height, math.ceil(cy+side/2))]\n    if result[2] <= result[0] or result[3] <= result[1]:\n        raise ValueError("Footprint lies outside image")\n    return result\n\n\ndef assign_splits(tile_ids, seed=42, demo_tiles=()):\n    """Split complete source tiles before crops. Demo tiles only enter test."""\n    tiles = sorted(set(tile_ids))\n    held_out = set(demo_tiles)\n    if not held_out <= set(tiles):\n        raise ValueError(f"Unknown demo tiles: {sorted(held_out-set(tiles))}")\n    candidates = sorted(set(tiles) - held_out)\n    if len(candidates) < 12:\n        raise ValueError("Need at least 12 non-demo source tiles for train/validation/test")\n    random.Random(seed).shuffle(candidates)\n    n_val = max(2, round(len(candidates) * 0.15))\n    n_test = max(2, round(len(candidates) * 0.15))\n    splits = {tile: "test" for tile in held_out}\n    splits.update({tile: "validation" for tile in candidates[:n_val]})\n    splits.update({tile: "test" for tile in candidates[n_val:n_val+n_test]})\n    splits.update({tile: "train" for tile in candidates[n_val+n_test:]})\n    return splits\n\n\ndef prepare(data_root, output, event="hurricane-harvey", seed=42, demo_tiles=()):\n    from PIL import Image\n    from shapely import wkt\n    root, output = Path(data_root).resolve(), Path(output)\n    label_files = sorted(root.rglob(f"{event}_*_post_disaster.json"))\n    if not label_files:\n        raise ValueError(f"No {event} post-disaster JSON labels under {root}. Attach xBD imagery AND original labels.")\n    image_index = {}\n    for image in sorted(root.rglob(f"{event}_*disaster.png")):\n        if image.name in image_index:\n            raise ValueError(f"Duplicate image basename: {image.name}; choose one dataset release")\n        image_index[image.name] = image\n    tile_labels = {}\n    for path in label_files:\n        tile = path.name.removesuffix("_post_disaster.json")\n        if tile in tile_labels:\n            raise ValueError(f"Duplicate label tile: {tile}; choose one dataset release")\n        tile_labels[tile] = path\n    split_map = assign_splits(tile_labels, seed, demo_tiles)\n    rows, sources, skipped = [], [], Counter()\n    for tile, label_path in tile_labels.items():\n        pre_path = image_index.get(tile + "_pre_disaster.png")\n        post_path = image_index.get(tile + "_post_disaster.png")\n        if not pre_path or not post_path:\n            raise ValueError(f"Missing before/after image for {tile}")\n        with Image.open(pre_path) as pre, Image.open(post_path) as post:\n            if pre.size != post.size:\n                raise ValueError(f"Unaligned image dimensions for {tile}")\n            width, height = pre.size\n        data = json.loads(label_path.read_text())\n        if not isinstance(data.get("features", {}).get("xy"), list):\n            raise ValueError(f"Missing xBD pixel polygons: {label_path.name}")\n        seen_ids = set()\n        for index, feature in enumerate(data["features"]["xy"]):\n            props = feature.get("properties", {})\n            label = props.get("subtype")\n            if label not in CLASSES:\n                skipped[str(label)] += 1\n                continue\n            geom = wkt.loads(feature["wkt"])\n            if geom.is_empty or not geom.is_valid or geom.geom_type not in ("Polygon", "MultiPolygon"):\n                skipped["invalid_polygon"] += 1\n                continue\n            building = str(props.get("uid", index))\n            if building in seen_ids:\n                raise ValueError(f"Duplicate building ID in {tile}: {building}")\n            seen_ids.add(building)\n            rows.append({"tile_id": tile, "building_id": building,\n                         "pre_image": str(pre_path.relative_to(root)),\n                         "post_image": str(post_path.relative_to(root)),\n                         "box": crop_bounds(geom.bounds, width, height),\n                         "label": CLASSES.index(label), "split": split_map[tile]})\n        sources.append({"tile_id": tile, "label_sha256": sha256(label_path),\n                        "pre_sha256": sha256(pre_path), "post_sha256": sha256(post_path)})\n    if not rows:\n        raise ValueError("No classified building polygons found")\n    counts = {part: dict(Counter(CLASSES[r["label"]] for r in rows if r["split"] == part))\n              for part in ("train", "validation", "test")}\n    if any(not counts[p] for p in counts):\n        raise ValueError(f"One or more splits have no classified buildings: {counts}")\n    missing = set(CLASSES) - set(counts["train"])\n    if missing:\n        raise ValueError(f"Training split lacks {sorted(missing)}. Choose a sufficiently representative dataset; do not silently relabel classes.")\n    manifest = {"schema_version": 1, "event": event, "seed": seed, "classes": CLASSES,\n                "image_size": IMAGE_SIZE, "split_unit": "source_tile", "rows": rows,\n                "sources": sources, "counts": counts, "skipped": dict(skipped),\n                "demo_tiles": sorted(demo_tiles),\n                "limitations": ["Classification on supplied building footprints, not localization.",\n                                "Tile-disjoint does not guarantee geographically disjoint scenes.",\n                                "Single-event estimates do not establish cross-disaster performance."]}\n    write_json(output, manifest)\n    return manifest\n\n\ndef metrics_from_predictions(labels, predictions):\n    """Fixed four-class macro F1, including zero-support classes explicitly."""\n    if len(labels) != len(predictions) or not labels:\n        raise ValueError("Metric inputs must be nonempty and equal length")\n    matrix = [[0] * len(CLASSES) for _ in CLASSES]\n    for actual, predicted in zip(labels, predictions):\n        if actual not in range(4) or predicted not in range(4):\n            raise ValueError("Class index out of range")\n        matrix[actual][predicted] += 1\n    per_class = {}\n    for i, name in enumerate(CLASSES):\n        tp, support = matrix[i][i], sum(matrix[i])\n        predicted_count = sum(row[i] for row in matrix)\n        precision = tp / predicted_count if predicted_count else 0.0\n        recall = tp / support if support else 0.0\n        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0\n        per_class[name] = {"precision": precision, "recall": recall, "f1": f1, "support": support}\n    return {"macro_f1": sum(v["f1"] for v in per_class.values())/4,\n            "accuracy": sum(matrix[i][i] for i in range(4))/len(labels),\n            "per_class": per_class, "confusion_matrix": matrix,\n            "absent_reference_classes": [k for k,v in per_class.items() if not v["support"]]}\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", required=True)\n    parser.add_argument("--output", default="manifest.json")\n    parser.add_argument("--event", default="hurricane-harvey")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--demo-tiles", nargs="*", default=[])\n    args = parser.parse_args()\n    result = prepare(args.data_root, args.output, args.event, args.seed, args.demo_tiles)\n    print(json.dumps({"buildings": len(result["rows"]), "tiles": len(result["sources"]),\n                      "counts": result["counts"], "skipped": result["skipped"]}, indent=2))\n')
sys.path.insert(0, "/kaggle/working")
from xbd_pipeline import prepare
manifest = prepare(DATA_ROOT, "/kaggle/working/manifest.json")
print(json.dumps({"buildings":len(manifest["rows"]), "tiles":len(manifest["sources"]), "counts":manifest["counts"], "skipped":manifest["skipped"]}, indent=2))


In [ ]:
from pathlib import Path
import subprocess, sys, shutil
Path("/kaggle/working/train_xbd.py").write_text('"""Train a paired RGB EfficientNet-B0 from random weights. No downloads."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nimport shutil\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom xbd_pipeline import CLASSES, IMAGE_SIZE, metrics_from_predictions, sha256, write_json\n\n\ndef make_model():\n    model = torchvision.models.efficientnet_b0(weights=None)\n    old = model.features[0][0]\n    model.features[0][0] = nn.Conv2d(6, old.out_channels, old.kernel_size,\n                                    old.stride, old.padding, bias=False)\n    nn.init.kaiming_normal_(model.features[0][0].weight, mode="fan_out")\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))\n    return model\n\n\ndef read_pair(root, row, augment=False):\n    crops = []\n    for key in ("pre_image", "post_image"):\n        path = (Path(root) / row[key]).resolve()\n        if not path.is_relative_to(Path(root).resolve()):\n            raise ValueError("Image path escapes data root")\n        with Image.open(path) as image:\n            image = image.convert("RGB").crop(row["box"]).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)\n            crops.append(np.asarray(image, dtype=np.float32)/255.0)\n    pair = np.concatenate(crops, axis=2)\n    return pair_tensor(pair, augment)\n\n\ndef pair_tensor(pair, augment=False):\n    if augment:\n        # Identical transforms preserve the temporal pairing.\n        pair = np.rot90(pair, random.randrange(4))\n        if random.random() < 0.5:\n            pair = np.flip(pair, axis=1)\n    pair = ((pair-0.5)/0.5).transpose(2,0,1).copy()\n    return torch.from_numpy(pair)\n\n\ndef cache_crops(root, rows, path):\n    """Decode each large satellite image only once, cache compact uint8 crops."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    required = len(rows)*IMAGE_SIZE*IMAGE_SIZE*6\n    if shutil.disk_usage(path.parent).free < required + 1024**3:\n        raise RuntimeError(f"Insufficient temporary storage for {required/1024**3:.1f} GiB crop cache")\n    cache = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8,\n                                    shape=(len(rows), IMAGE_SIZE, IMAGE_SIZE, 6))\n    tiles = defaultdict(list)\n    for index,row in enumerate(rows):\n        tiles[row["tile_id"]].append((index,row))\n    lookup = {}\n    for tile_number,(tile,group) in enumerate(tiles.items()):\n        row = group[0][1]\n        with Image.open(Path(root)/row["pre_image"]) as before, Image.open(Path(root)/row["post_image"]) as after:\n            before, after = before.convert("RGB"), after.convert("RGB")\n            for index,row in group:\n                crops = [np.asarray(image.crop(row["box"]).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR))\n                         for image in (before, after)]\n                cache[index] = np.concatenate(crops,axis=2)\n                lookup[(row["tile_id"],row["building_id"])] = index\n        if tile_number % 25 == 0:\n            print(json.dumps({"status":"preparing_crops", "tile":tile_number, "total_tiles":len(tiles)}), flush=True)\n    cache.flush()\n    del cache\n    return lookup\n\n\nclass BuildingPairs(Dataset):\n    def __init__(self, root, rows, augment=False, cache_path=None, lookup=None):\n        self.root, self.rows, self.augment = root, rows, augment\n        self.cache_path, self.lookup, self.cache = cache_path, lookup, None\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        if self.cache_path:\n            if self.cache is None:\n                self.cache = np.load(self.cache_path, mmap_mode="r")\n            value = self.cache[self.lookup[(row["tile_id"],row["building_id"])]]\n            return pair_tensor(value.astype(np.float32)/255.0, self.augment), row["label"]\n        return read_pair(self.root, row, self.augment), row["label"]\n\n\ndef evaluate(model, loader, device):\n    model.eval()\n    labels, predictions, probabilities = [], [], []\n    with torch.inference_mode():\n        for features, target in loader:\n            probs = torch.softmax(model(features.to(device)), dim=1).cpu()\n            labels.extend(target.tolist())\n            predictions.extend(probs.argmax(1).tolist())\n            probabilities.extend(probs.tolist())\n    return metrics_from_predictions(labels, predictions), probabilities\n\n\ndef train(args):\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.set_num_threads(min(4, torch.get_num_threads()))\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda" and not args.allow_cpu:\n        raise RuntimeError("No GPU visible. Select only an available free Kaggle GPU; do not purchase compute.")\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    if (output / "best.pt").exists():\n        raise ValueError("Output already has a model. Use a new output directory; existing results are not overwritten.")\n    manifest = json.loads(Path(args.manifest).read_text())\n    if manifest["classes"] != CLASSES or manifest["split_unit"] != "source_tile":\n        raise ValueError("Unexpected label order or split policy")\n    split_rows = {part: [r for r in manifest["rows"] if r["split"] == part]\n                  for part in ("train", "validation", "test")}\n    tile_sets = {part: {r["tile_id"] for r in rows} for part, rows in split_rows.items()}\n    for a,b in (("train","validation"),("train","test"),("validation","test")):\n        if tile_sets[a] & tile_sets[b]:\n            raise ValueError("Source-tile leakage detected")\n    if any(not rows for rows in split_rows.values()):\n        raise ValueError("Empty data split")\n    write_json(output / "status.json", {"status":"preparing_crops"})\n    lookup = cache_crops(args.data_root, manifest["rows"], args.crop_cache)\n    loaders = {part: DataLoader(BuildingPairs(args.data_root, rows, part == "train", args.crop_cache, lookup),\n                               batch_size=args.batch_size, shuffle=part == "train",\n                               num_workers=args.workers, pin_memory=device.type == "cuda")\n               for part, rows in split_rows.items()}\n    counts = np.bincount([r["label"] for r in split_rows["train"]], minlength=4)\n    if np.any(counts == 0):\n        raise ValueError("Every damage class must occur in the training split")\n    weights = counts.astype(float) ** -0.5\n    weights /= weights.mean()\n    model = make_model().to(device)\n    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=1e-4)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)\n    config = {"schema_version": 1, "architecture": "efficientnet_b0_6channel",\n              "initialization": "random", "pretrained_weights": None,\n              "classes": CLASSES, "image_size": IMAGE_SIZE,\n              "channel_order": ["pre_R","pre_G","pre_B","post_R","post_G","post_B"],\n              "normalization": {"mean": [0.5]*6, "std": [0.5]*6},\n              "manifest_sha256": sha256(args.manifest), "seed": args.seed,\n              "event": manifest["event"], "device": str(device),\n              "torch_version": str(torch.__version__), "torchvision_version": str(torchvision.__version__),\n              "hyperparameters": vars(args), "limitations": manifest["limitations"]}\n    write_json(output / "model_config.json", config)\n    write_json(output / "manifest.json", manifest)\n    start, best_score, best_epoch, stale = time.monotonic(), -1.0, 0, 0\n    history = []\n    for epoch in range(1, args.epochs+1):\n        model.train()\n        running_loss, seen = 0.0, 0\n        for batch_index, (features, target) in enumerate(loaders["train"]):\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(features.to(device))\n            loss = loss_fn(logits, target.to(device))\n            if not torch.isfinite(loss):\n                raise RuntimeError("Non-finite training loss")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            optimizer.step()\n            seen += len(target)\n            running_loss += loss.item()*len(target)\n            if batch_index % 25 == 0:\n                write_json(output / "status.json", {"status":"training", "epoch":epoch,\n                           "batch":batch_index, "elapsed_seconds":time.monotonic()-start})\n            if time.monotonic()-start > args.max_hours*3600:\n                break\n        val_metrics, _ = evaluate(model, loaders["validation"], device)\n        record = {"epoch":epoch, "train_loss":running_loss/max(seen,1),\n                  "validation":val_metrics, "elapsed_seconds":time.monotonic()-start}\n        history.append(record)\n        write_json(output / "history.json", history)\n        print(json.dumps(record), flush=True)\n        if val_metrics["macro_f1"] > best_score:\n            best_score, best_epoch, stale = val_metrics["macro_f1"], epoch, 0\n            torch.save({"state_dict": {k:v.detach().cpu() for k,v in model.state_dict().items()},\n                        "epoch": epoch, "validation_macro_f1":best_score}, output / "best.pt")\n        else:\n            stale += 1\n        scheduler.step()\n        if stale >= args.patience or time.monotonic()-start > args.max_hours*3600:\n            break\n    checkpoint = torch.load(output / "best.pt", map_location=device, weights_only=True)\n    model.load_state_dict(checkpoint["state_dict"])\n    # Test is opened once, after selection by validation only.\n    test_metrics, probabilities = evaluate(model, loaders["test"], device)\n    final = {"best_epoch": best_epoch, "validation_macro_f1": best_score, "test": test_metrics,\n             "elapsed_seconds": time.monotonic()-start, "completed_epochs":len(history),\n             "majority_baseline_test": metrics_from_predictions(\n                 [r["label"] for r in split_rows["test"]],\n                 [int(counts.argmax())]*len(split_rows["test"]))}\n    write_json(output / "metrics.json", final)\n    predictions = []\n    model_hash = sha256(output / "best.pt")\n    for row, probs in zip(split_rows["test"], probabilities):\n        predictions.append({"tile_id":row["tile_id"], "building_id":row["building_id"],\n                            "box":row["box"], "damage_class":CLASSES[int(np.argmax(probs))],\n                            "probabilities":dict(zip(CLASSES, probs)), "state":"estimated",\n                            "source":"custom_model", "split":"test",\n                            "reference_label":CLASSES[row["label"]],\n                            "model_sha256":model_hash})\n    write_json(output / "heldout_predictions.json", predictions)\n    # Preserve three truly held-out image pairs for the demo, with attribution.\n    demo_tiles = sorted(tile_sets["test"])[:3]\n    demo_dir = output / "demo_images"\n    demo_dir.mkdir(exist_ok=True)\n    for tile in demo_tiles:\n        row = next(row for row in split_rows["test"] if row["tile_id"] == tile)\n        for key in ("pre_image", "post_image"):\n            shutil.copyfile(Path(args.data_root) / row[key], demo_dir / Path(row[key]).name)\n    write_json(output / "demo_tiles.json", demo_tiles)\n    write_json(output / "attribution.json", {\n        "dataset":"xBD / xView2", "authors":"Gupta et al.; DIU, CMU SEI and collaborators; Maxar imagery",\n        "source":"https://www.kaggle.com/datasets/tunguz/xview2-challenge-dataset-train-and-test",\n        "version":3, "license":"CC BY-NC-SA 4.0",\n        "license_url":"https://creativecommons.org/licenses/by-nc-sa/4.0/",\n        "paper":"https://arxiv.org/abs/1911.09296",\n        "use":"Noncommercial hackathon research prototype; preserve attribution and applicable share-alike terms."})\n    write_json(output / "status.json", {"status":"completed", "best_epoch":best_epoch,\n               "elapsed_seconds":time.monotonic()-start, "model_sha256":model_hash})\n    write_json(output / "checksums.json", {str(p.relative_to(output)):sha256(p) for p in output.rglob("*") if p.is_file() and p.name != "checksums.json"})\n    print(json.dumps({"status":"completed", "output":str(output), **final}), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", required=True)\n    parser.add_argument("--manifest", required=True)\n    parser.add_argument("--output", default="/kaggle/working/xbd_model")\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--learning-rate", type=float, default=1e-3)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--patience", type=int, default=4)\n    parser.add_argument("--max-hours", type=float, default=2.0)\n    parser.add_argument("--allow-cpu", action="store_true", help="For local smoke tests only")\n    parser.add_argument("--crop-cache", default="/kaggle/temp/xbd_crops.npy")\n    arguments = parser.parse_args()\n    if min(arguments.epochs, arguments.batch_size, arguments.patience, arguments.max_hours) <= 0:\n        parser.error("Training limits must be positive")\n    try:\n        train(arguments)\n    except Exception as exc:\n        write_json(Path(arguments.output) / "status.json", {"status":"failed", "error":str(exc)})\n        raise\n')
Path("/kaggle/working/infer_xbd.py").write_text('"""One paired, pre-cropped building inference. Output JSON to stdout."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom xbd_pipeline import CLASSES, sha256\nfrom train_xbd import make_model, read_pair\n\n\ndef infer(model_dir, root, pre_image, post_image, box):\n    model_dir = Path(model_dir)\n    config = json.loads((model_dir / "model_config.json").read_text())\n    if config["classes"] != CLASSES or config["architecture"] != "efficientnet_b0_6channel":\n        raise ValueError("Unsupported model contract")\n    model = make_model()\n    checkpoint = torch.load(model_dir / "best.pt", map_location="cpu", weights_only=True)\n    model.load_state_dict(checkpoint["state_dict"])\n    model.eval()\n    with torch.inference_mode():\n        row = {"pre_image":pre_image, "post_image":post_image, "box":box}\n        probs = torch.softmax(model(read_pair(root, row)[None]), 1)[0].tolist()\n    return {"damage_class":CLASSES[max(range(4), key=lambda i:probs[i])],\n            "probabilities":dict(zip(CLASSES, probs)), "state":"estimated", "source":"custom_model",\n            "model_sha256":sha256(model_dir / "best.pt"), "calibrated":False,\n            "limitations":["Building-footprint crop classification only.",\n                           "Not evidence of personal safety, road access or shelter suitability."]}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model-dir", required=True)\n    parser.add_argument("--data-root", required=True)\n    parser.add_argument("--pre-image", required=True)\n    parser.add_argument("--post-image", required=True)\n    parser.add_argument("--box", nargs=4, type=int, required=True)\n    args = parser.parse_args()\n    print(json.dumps(infer(args.model_dir, args.data_root, args.pre_image, args.post_image, args.box)))\n')
subprocess.run([sys.executable, "-u", "/kaggle/working/train_xbd.py", "--data-root", DATA_ROOT, "--manifest", "/kaggle/working/manifest.json", "--output", "/kaggle/working/xbd_model", "--epochs", "12", "--max-hours", "2", "--batch-size", "32", "--workers", "2"], check=True)
shutil.copyfile("/kaggle/working/train_xbd.py", "/kaggle/working/xbd_model/train_xbd.py")
shutil.copyfile("/kaggle/working/xbd_pipeline.py", "/kaggle/working/xbd_model/xbd_pipeline.py")
shutil.copyfile("/kaggle/working/infer_xbd.py", "/kaggle/working/xbd_model/infer_xbd.py")
shutil.make_archive("/kaggle/working/utd_xbd_model", "zip", "/kaggle/working/xbd_model")
print("MODEL_BUNDLE_READY: utd_xbd_model.zip")
